In [ ]:
"""
六模型 PDE 求解对比可视化脚本（Poisson 方程）- 学术排版版
=======================================================
支持 PFDv1/PFDv2 从完整检查点或分别加载子域坐标

生成两张图：
  Figure 1 — 收敛曲线对比（左: Total Loss，右: Rel. L2 Error）
  Figure 2 — 解的可视化（学术排版，大字号）
             布局（3 行 × N 列）：
               Row 0 │ Model1 预测 │ Model2 预测 │ Model3 预测 │ … │
               Row 1 │ Model1 误差 │ Model2 误差 │ Model3 误差 │ … │
               Row 2 │ Section y=0 │ Section x=0 │ （跨列合并）│

使用方式：
  python plot_pde_comparison.py
"""

import os
import warnings
import numpy as np
import torch
import torch.nn as nn
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import ticker
from matplotlib.path import Path as MplPath
from scipy.spatial import ConvexHull

warnings.filterwarnings("ignore")

# =============================================================================
# ★ 学术排版字号规范（大字号）
# =============================================================================
FS_SUPTITLE   = 16
FS_COL_HEADER = 14   # 列顶部模型名
FS_ROW_LABEL  = 14   # 左侧行标签
FS_TITLE      = 13   # 子图标题
FS_LABEL      = 13   # 坐标轴标签
FS_TICK       = 12   # 刻度数字
FS_LEGEND     = 11   # 图例
FS_CBAR       = 11   # colorbar 刻度

plt.rcParams.update({
    "font.size":        FS_TICK,
    "axes.titlesize":   FS_TITLE,
    "axes.labelsize":   FS_LABEL,
    "xtick.labelsize":  FS_TICK,
    "ytick.labelsize":  FS_TICK,
    "legend.fontsize":  FS_LEGEND,
    "figure.dpi":       150,
})

# =============================================================================
# ── 全局配置 ──────────────────────────────────────────────────────────────────
# =============================================================================
MODEL_STYLE = {
    "PINN":       dict(color="#6B7280", ls="-",  marker="o",  label="PINN"),
    "MscaleDNN":  dict(color="#F59E0B", ls="-",  marker="s",  label="MscaleDNN"),
    "Test-A":     dict(color="#10B981", ls="--", marker="^",  label="Test-A"),
    "Test-B":     dict(color="#3B82F6", ls="--", marker="D",  label="Test-B"),
    "PFDv1":      dict(color="#EF4444", ls="-",  marker="P",  label="PFDv1"),
    "PFDv2":      dict(color="#8B5CF6", ls="-",  marker="*",  label="PFDv2"),
}

MU = 30.0
DEVICE = torch.device("cpu")
pi = torch.tensor(np.pi, dtype=torch.float64)


# =============================================================================
# ── 辅助函数：统一 colorbar 格式 ──────────────────────────────────────────────
# =============================================================================
def _add_cbar(fig, cf, ax, fmt="%.2f"):
    cb = fig.colorbar(cf, ax=ax, fraction=0.046, pad=0.04,
                      format=ticker.FormatStrFormatter(fmt))
    cb.ax.tick_params(labelsize=FS_CBAR)
    return cb


# =============================================================================
# ── 精确解函数 ────────────────────────────────────────────────────────────────
# =============================================================================
def g_func(x):
    return np.exp(-x ** 2) * np.sin(MU * x ** 2)

def u_exact_np(xy):
    """xy: (N, 2) numpy → (N,) numpy"""
    x, y = xy[:, 0], xy[:, 1]
    return g_func(x) + g_func(y)


# =============================================================================
# ── 可视化网格 ────────────────────────────────────────────────────────────────
# =============================================================================
def make_grid(N=200):
    gx = np.linspace(-1, 1, N)
    gy = np.linspace(-1, 1, N)
    GX, GY = np.meshgrid(gx, gy, indexing="ij")
    xy_flat = np.stack([GX.ravel(), GY.ravel()], axis=1)
    u_exact = u_exact_np(xy_flat).reshape(N, N)
    return gx, gy, xy_flat, u_exact, GX, GY


# =============================================================================
# ── 网络定义（与训练脚本保持一致）─────────────────────────────────────────────
# =============================================================================

class PINN(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=6):
        super().__init__()
        layers = [nn.Linear(2, hidden_dim), nn.Tanh()]
        for _ in range(num_layers - 2):
            layers += [nn.Linear(hidden_dim, hidden_dim), nn.Tanh()]
        layers.append(nn.Linear(hidden_dim, 1))
        self.net = nn.Sequential(*layers)
    def forward(self, xy):
        return self.net(xy)


class ScaleSubNet(nn.Module):
    def __init__(self, scale_factor, hidden_dim, num_layers):
        super().__init__()
        self.scale = scale_factor
        layers, in_dim = [], 2
        for _ in range(num_layers):
            layers.append(nn.Linear(in_dim, hidden_dim))
            in_dim = hidden_dim
        self.layers = nn.ModuleList(layers)
    def forward(self, xy):
        x = xy * self.scale
        for i, layer in enumerate(self.layers):
            x = layer(x)
            if i < len(self.layers) - 1:
                x = torch.sin(x) if i % 2 == 0 else torch.tanh(x)
        return x

class MscaleDNN(nn.Module):
    def __init__(self, scales=None, hidden_dim=150, sub_layers=4):
        super().__init__()
        if scales is None:
            scales = [1, 2, 4, 8, 16, 32]
        self.subnets = nn.ModuleList([
            ScaleSubNet(s, hidden_dim, sub_layers) for s in scales
        ])
        fuse_in = hidden_dim * len(scales)
        self.fuse = nn.Sequential(
            nn.Linear(fuse_in, hidden_dim), nn.Tanh(),
            nn.Linear(hidden_dim, 1))
    def forward(self, xy):
        return self.fuse(torch.cat([s(xy) for s in self.subnets], dim=-1))


class FixedFourierEmbed2D(nn.Module):
    def __init__(self, scales):
        super().__init__()
        self.register_buffer("scales", torch.tensor(scales, dtype=torch.float64))
    def forward(self, xy):
        feats = []
        for s in self.scales:
            feats += [
                torch.sin(2 * pi * s * xy[:, 0:1]),
                torch.cos(2 * pi * s * xy[:, 0:1]),
                torch.sin(2 * pi * s * xy[:, 1:2]),
                torch.cos(2 * pi * s * xy[:, 1:2]),
            ]
        return torch.cat(feats, dim=-1)

class APINN2D_Fixed(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=4, scales=(1, 2, 4)):
        super().__init__()
        self.embed = FixedFourierEmbed2D(scales)
        in_dim = 2 + 4 * len(scales)
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, 1))
    def forward(self, xy):
        h = torch.cat([xy, self.embed(xy)], dim=-1)
        for layer in self.layers[:-1]:
            h = torch.tanh(layer(h))
        return self.layers[-1](h)


class AdaptiveFourierEmbed2D(nn.Module):
    def __init__(self, init_scales):
        super().__init__()
        log_s = torch.log(torch.tensor(init_scales, dtype=torch.float64))
        self.log_scales = nn.Parameter(log_s)
    def forward(self, xy):
        scales = torch.exp(self.log_scales)
        feats = []
        for s in scales:
            feats += [
                torch.sin(2 * pi * s * xy[:, 0:1]),
                torch.cos(2 * pi * s * xy[:, 0:1]),
                torch.sin(2 * pi * s * xy[:, 1:2]),
                torch.cos(2 * pi * s * xy[:, 1:2]),
            ]
        return torch.cat(feats, dim=-1)

class APINN2D_Adaptive(nn.Module):
    def __init__(self, hidden_dim=128, num_layers=4, scales=(1, 2, 4)):
        super().__init__()
        self.embed = AdaptiveFourierEmbed2D(scales)
        in_dim = 2 + 4 * len(scales)
        self.layers = nn.ModuleList()
        self.layers.append(nn.Linear(in_dim, hidden_dim))
        for _ in range(num_layers - 2):
            self.layers.append(nn.Linear(hidden_dim, hidden_dim))
        self.layers.append(nn.Linear(hidden_dim, 1))
    def forward(self, xy):
        h = torch.cat([xy, self.embed(xy)], dim=-1)
        for layer in self.layers[:-1]:
            h = torch.tanh(layer(h))
        return self.layers[-1](h)


class ConvexHullSubdomain:
    def __init__(self, hull_vertices, decay=15.0):
        self.vertices_np = hull_vertices
        self.decay = decay
        verts = np.vstack([hull_vertices, hull_vertices[0]])
        edges = verts[1:] - verts[:-1]
        norms = np.linalg.norm(edges, axis=1, keepdims=True)
        self.edge_dirs = edges / (norms + 1e-12)
        self.edge_start = verts[:-1]
        self.edge_len = norms.flatten()
        self.mpl_path = MplPath(hull_vertices)
        self.center = hull_vertices.mean(axis=0)
        self.radius = np.max(np.linalg.norm(hull_vertices - self.center, axis=1))
    def dist_to_hull_boundary(self, xy_np):
        min_dist = np.full(len(xy_np), np.inf)
        for k in range(len(self.edge_dirs)):
            p0, d, L = self.edge_start[k], self.edge_dirs[k], self.edge_len[k]
            vec = xy_np - p0
            t = np.clip(vec @ d, 0, L)
            proj = p0 + t[:, None] * d
            dist = np.linalg.norm(xy_np - proj, axis=1)
            min_dist = np.minimum(min_dist, dist)
        inside = self.mpl_path.contains_points(xy_np).astype(float)
        return np.where(inside > 0.5, min_dist, -min_dist)
    def support_function(self, xy):
        xy_np = xy.detach().cpu().numpy()
        d_np = self.dist_to_hull_boundary(xy_np).astype(np.float64)
        d_t = torch.tensor(d_np, dtype=torch.float64, device=xy.device).unsqueeze(-1)
        inside = (d_t > 0).float()
        phi = inside * torch.sigmoid(self.decay * d_t) + (1 - inside) * torch.exp(self.decay * d_t)
        return phi.clamp(0.0, 1.0)


class PFDNet2D_TestA(nn.Module):
    def __init__(self, apinn_low, high_scales, hidden_dim=128, num_layers=4):
        super().__init__()
        self.apinn_low = apinn_low
        self.high_net = APINN2D_Fixed(hidden_dim, num_layers, high_scales)
    def forward(self, xy):
        return self.apinn_low(xy) + self.high_net(xy)


class PFDNet2D_V1(nn.Module):
    def __init__(self, apinn_low, subdomains, high_scales, hidden_dim=128, num_layers=4):
        super().__init__()
        self.apinn_low = apinn_low
        self.subdomains = subdomains
        self.high_nets = nn.ModuleList([
            APINN2D_Fixed(hidden_dim, num_layers, high_scales)
            for _ in subdomains
        ])
    def forward(self, xy):
        u = self.apinn_low(xy)
        for sd, net in zip(self.subdomains, self.high_nets):
            u = u + sd.support_function(xy) * net(xy)
        return u


class PFDNet2D_V2(nn.Module):
    def __init__(self, apinn_low, subdomains, high_scales, hidden_dim=128, num_layers=4):
        super().__init__()
        self.apinn_low = apinn_low
        self.subdomains = subdomains
        self.high_nets = nn.ModuleList([
            APINN2D_Adaptive(hidden_dim, num_layers, high_scales)
            for _ in subdomains
        ])
    def forward(self, xy):
        u = self.apinn_low(xy)
        for sd, net in zip(self.subdomains, self.high_nets):
            u = u + sd.support_function(xy) * net(xy)
        return u


# =============================================================================
# ── 模型加载辅助（支持 V1/V2 的子域加载）─────────────────────────────────────
# =============================================================================
def _try_load(model, path):
    if not os.path.exists(path):
        return False
    try:
        state = torch.load(path, map_location="cpu")
        model.load_state_dict(state)
        model.eval()
        return True
    except Exception as e:
        print(f"  WARNING: failed to load {path}: {e}")
        return False

def _load_subdomains(checkpoint_dir):
    """从目录加载子域顶点坐标"""
    n_path = os.path.join(checkpoint_dir, "n_subdomains.npy")
    if os.path.exists(n_path):
        n_subdomains = int(np.load(n_path))
        subdomains = []
        for i in range(n_subdomains):
            verts_path = os.path.join(checkpoint_dir, f"subdomain_{i}_vertices.npy")
            if os.path.exists(verts_path):
                verts = np.load(verts_path)
                subdomains.append(ConvexHullSubdomain(verts))
        return subdomains
    return None

def _load_full_checkpoint(checkpoint_dir, model_name):
    """尝试加载完整检查点（包含子域和超参）"""
    full_path = os.path.join(checkpoint_dir, f"{model_name}_full.pt")
    if os.path.exists(full_path):
        try:
            ckpt = torch.load(full_path, map_location="cpu", weights_only=False)
            return ckpt
        except:
            pass
    return None


def build_model_pinn():
    m = PINN(hidden_dim=128, num_layers=6).double()
    ok = _try_load(m, "checkpoints_pinn_poisson/pinn_poisson_weights.pt")
    return m if ok else None

def build_model_mscale():
    m = MscaleDNN(scales=[1, 2, 4, 8, 16, 32], hidden_dim=150, sub_layers=4).double()
    ok = _try_load(m, "checkpoints_mscalednn_pinn_poisson/mscalednn_pinn_weights.pt")
    return m if ok else None

def build_model_testA():
    apinn_low = APINN2D_Fixed(128, 4, (1, 2, 4)).double()
    m = PFDNet2D_TestA(apinn_low, (8, 16, 32), 128, 4).double()
    ok = _try_load(m, "checkpoints_test_A_pinn/testA_pinn_weights.pt")
    return m if ok else None

def build_model_testB():
    m = APINN2D_Fixed(128, 4, (1, 2, 4)).double()
    ok = _try_load(m, "checkpoints_test_B_pde/testB_pde_weights.pt")
    return m if ok else None

def build_model_pfdv1():
    checkpoint_dir = "checkpoints_pfdv1_pinn_poisson"
    weights_path = os.path.join(checkpoint_dir, "pfdv1_pinn_weights.pt")

    if not os.path.exists(weights_path):
        return None

    ckpt = _load_full_checkpoint(checkpoint_dir, "pfdv1")
    if ckpt is not None:
        subdomains = [ConvexHullSubdomain(verts) for verts in ckpt['subdomain_vertices']]
        apinn_low = APINN2D_Fixed(
            ckpt.get('hidden_dim', 128),
            ckpt.get('num_layers', 4),
            ckpt.get('low_scales', (1, 2, 4))
        ).double()
        m = PFDNet2D_V1(
            apinn_low, subdomains,
            ckpt.get('high_scales', (8, 16, 32)),
            ckpt.get('hidden_dim', 128),
            ckpt.get('num_layers', 4)
        ).double()
        m.load_state_dict(ckpt['model_state'])
        m.eval()
        return m

    subdomains = _load_subdomains(checkpoint_dir)
    if subdomains is None:
        verts = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=float)
        subdomains = [ConvexHullSubdomain(verts)]

    apinn_low = APINN2D_Fixed(128, 4, (1, 2, 4)).double()
    m = PFDNet2D_V1(apinn_low, subdomains, (8, 16, 32), 128, 4).double()
    ok = _try_load(m, weights_path)
    return m if ok else None

def build_model_pfdv2():
    checkpoint_dir = "checkpoints_pfdv2_pinn_poisson"
    weights_path = os.path.join(checkpoint_dir, "pfdv2_pinn_weights.pt")

    if not os.path.exists(weights_path):
        return None

    ckpt = _load_full_checkpoint(checkpoint_dir, "pfdv2")
    if ckpt is not None:
        subdomains = [ConvexHullSubdomain(verts) for verts in ckpt['subdomain_vertices']]
        apinn_low = APINN2D_Adaptive(
            ckpt.get('hidden_dim', 128),
            ckpt.get('num_layers', 4),
            ckpt.get('low_scales', (1, 2, 4))
        ).double()
        m = PFDNet2D_V2(
            apinn_low, subdomains,
            ckpt.get('high_scales', (8, 16, 32)),
            ckpt.get('hidden_dim', 128),
            ckpt.get('num_layers', 4)
        ).double()
        m.load_state_dict(ckpt['model_state'])
        m.eval()
        return m

    subdomains = _load_subdomains(checkpoint_dir)
    if subdomains is None:
        verts = np.array([[-1, -1], [1, -1], [1, 1], [-1, 1]], dtype=float)
        subdomains = [ConvexHullSubdomain(verts)]

    apinn_low = APINN2D_Adaptive(128, 4, (1, 2, 4)).double()
    m = PFDNet2D_V2(apinn_low, subdomains, (8, 16, 32), 128, 4).double()
    ok = _try_load(m, weights_path)
    return m if ok else None


# =============================================================================
# ── 曲线数据加载（直接读取，无缩放）───────────────────────────────────────────
# =============================================================================
CURVE_FILES = {
    "PINN":      "checkpoints_pinn_poisson/pinn_poisson_curves.npz",
    "MscaleDNN": "checkpoints_mscalednn_pinn_poisson/mscalednn_pinn_curves.npz",
    "Test-A":    "checkpoints_test_A_pinn/testA_pinn_curves.npz",
    "Test-B":    "checkpoints_test_B_pde/testB_pde_curves.npz",
    "PFDv1":     "checkpoints_pfdv1_pinn_poisson/pfdv1_pinn_curves.npz",
    "PFDv2":     "checkpoints_pfdv2_pinn_poisson/pfdv2_pinn_curves.npz",
}

def load_curves():
    """加载曲线数据（无任何缩放）"""
    curves = {}
    for name, path in CURVE_FILES.items():
        if os.path.exists(path):
            d = np.load(path)
            total_loss = None
            for key in ["total_loss", "total", "train_loss"]:
                if key in d:
                    total_loss = d[key]
                    break

            l2_error = None
            for key in ["l2_error", "l2"]:
                if key in d:
                    l2_error = d[key]
                    break

            epochs = d["epochs"]

            curves[name] = {
                "epochs": epochs,
                "total_loss": total_loss,
                "l2_error": l2_error,
            }
        else:
            print(f"  not found: {path}")
    return curves


# =============================================================================
# ── Figure 1: 收敛曲线对比（保持原有风格，仅调整字号）────────────────────────
# =============================================================================
def plot_figure1(curves, save_path="figure1_pde_convergence.pdf"):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    items = []
    for name, data in curves.items():
        if name not in MODEL_STYLE:
            continue
        sty = MODEL_STYLE[name]
        ep = data["epochs"]
        loss = data["total_loss"] if data["total_loss"] is not None and len(data["total_loss"]) > 0 else None
        l2 = data["l2_error"] if data["l2_error"] is not None and len(data["l2_error"]) > 0 else None
        final_loss = loss[-1] if loss is not None else None
        final_l2 = l2[-1] if l2 is not None else None
        items.append((name, sty, ep, loss, l2, final_loss, final_l2))

    items_left = sorted([it for it in items if it[5] is not None], key=lambda x: x[5])
    for name, sty, ep, loss, l2, final_loss, final_l2 in items_left:
        ax1.semilogy(ep, loss, color=sty["color"], ls=sty["ls"], lw=1.5, label=sty['label'])

    items_right = sorted([it for it in items if it[6] is not None], key=lambda x: x[6])
    for name, sty, ep, loss, l2, final_loss, final_l2 in items_right:
        ax2.semilogy(ep, l2, color=sty["color"], ls=sty["ls"], lw=1.5,
                     label=f"{sty['label']} (final={final_l2:.4f})")

    ax1.set_xlabel("Epoch", fontsize=FS_LABEL)
    ax1.set_ylabel("Total Loss (MSE)", fontsize=FS_LABEL)
    ax1.set_title("Training Total Loss", fontsize=FS_TITLE)
    ax1.legend(fontsize=FS_LEGEND, ncol=2)
    ax1.grid(True, alpha=0.3)
    ax1.tick_params(labelsize=FS_TICK)

    ax2.set_xlabel("Epoch", fontsize=FS_LABEL)
    ax2.set_ylabel("Relative $L_2$ Error", fontsize=FS_LABEL)
    ax2.set_title("Test Relative $L_2$ Error", fontsize=FS_TITLE)
    ax2.legend(fontsize=FS_LEGEND, ncol=2)
    ax2.grid(True, alpha=0.3)
    ax2.tick_params(labelsize=FS_TICK)

    plt.tight_layout()
    fig.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print(f"Figure 1 saved: {save_path}")


# =============================================================================
# ── Figure 2: 解的可视化（学术排版，大字号）───────────────────────────────────
#    布局（3 行 × N 列）：
#      Row 0 │ 预测图（jet colormap）
#      Row 1 │ 误差图（jet colormap）
#      Row 2 │ 截面图（y=0 左，x=0 右，跨列合并）
# =============================================================================
def plot_figure2(models, save_path="figure2_pde_solution.pdf"):
    N_GRID = 150
    gx, gy, xy_flat, u_exact, X2D, Y2D = make_grid(N_GRID)

    # 计算各模型预测（直接使用原始输出）
    preds = {}
    scaled_l2s = {}

    for name, model in models.items():
        if model is None:
            continue
        xy_t = torch.tensor(xy_flat, dtype=torch.float64)
        with torch.no_grad():
            out = model(xy_t).squeeze(-1).numpy()
        pred = out.reshape(N_GRID, N_GRID)
        preds[name] = pred

        # 计算 L2 误差
        err = np.abs(pred - u_exact)
        scaled_l2s[name] = np.sqrt(np.mean(err**2) / np.mean(u_exact**2))

    model_names = list(preds.keys())
    N = len(model_names)
    if N == 0:
        print("No model weights found, skipping Figure 2.")
        return

    errs = {name: np.abs(preds[name] - u_exact) for name in model_names}

    vmin_sol = float(u_exact.min())
    vmax_sol = float(u_exact.max())
    emax = max(e.max() for e in errs.values())

    # ── Figure 布局（3 行 × N 列）───────────────────────────────────────────────
    col_w = 3.4                     # 每列宽度（英寸）
    row_h = [3.2, 3.2, 3.2]         # 各行高度（英寸）
    top_margin = 0.9
    fig_w = N * col_w + 0.8
    fig_h = sum(row_h) + top_margin + 0.8

    fig = plt.figure(figsize=(fig_w, fig_h))

    gs = gridspec.GridSpec(
        3, N,
        figure=fig,
        height_ratios=row_h,
        hspace=0.15,
        wspace=0.3,
        left=0.10,
        right=0.90,
        top=0.85,
        bottom=0.08,
    )

    # ── 添加行标签（左侧）─────────────────────────────────────────────────────
    row_labels = ["Prediction", "Absolute Error", "Section Plot"]
    for i, label in enumerate(row_labels):
        pos = gs[i, 0].get_position(fig)
        y_center = (pos.y0 + pos.y1) / 2
        fig.text(
            0.05, y_center, label,
            va="center", ha="center",
            fontsize=FS_ROW_LABEL,
            fontweight="bold",
            rotation=90,
        )

    # ── 添加列标签（顶部：模型名 + ℓ₂ 误差）───────────────────────────────────
    for j, name in enumerate(model_names):
        pos = gs[0, j].get_position(fig)
        x_center = (pos.x0 + pos.x1) / 2
        y_top = pos.y1 + 0.02

        fig.text(
            x_center, y_top + 0,
            MODEL_STYLE[name]["label"],
            va="bottom", ha="center",
            fontsize=FS_COL_HEADER,
            fontweight="bold",
        )
        fig.text(
            x_center, y_top - 0.02,
            f"$\\ell_2 = {scaled_l2s[name]:.4f}$",
            va="top", ha="center",
            fontsize=FS_COL_HEADER - 2,
            fontstyle="italic",
        )

    # ─────────────────────────────────────────────────────────────────────────
    # Row 0：各模型预测（jet colormap）
    # ─────────────────────────────────────────────────────────────────────────
    for j, name in enumerate(model_names):
        ax = fig.add_subplot(gs[0, j])
        cf = ax.contourf(X2D, Y2D, preds[name],
                         levels=60, cmap="jet",
                         vmin=vmin_sol, vmax=vmax_sol)
        _add_cbar(fig, cf, ax)
        ax.set_aspect("equal")
        ax.tick_params(labelsize=FS_TICK)

        if j > 0:
            ax.tick_params(labelleft=False)

        if j == 0:
            ax.set_ylabel("$y$", fontsize=FS_LABEL)
        else:
            ax.set_ylabel("")

        ax.set_xlabel("")

    # ─────────────────────────────────────────────────────────────────────────
    # Row 1：各模型绝对误差（jet colormap）
    # ─────────────────────────────────────────────────────────────────────────
    for j, name in enumerate(model_names):
        ax = fig.add_subplot(gs[1, j])
        cf = ax.contourf(X2D, Y2D, errs[name],
                         levels=50, cmap="jet",
                         vmin=0, vmax=emax)
        _add_cbar(fig, cf, ax, fmt="%.3f")
        ax.set_aspect("equal")
        ax.tick_params(labelsize=FS_TICK)

        if j > 0:
            ax.tick_params(labelleft=False)

        if j == 0:
            ax.set_ylabel("$y$", fontsize=FS_LABEL)
        else:
            ax.set_ylabel("")

        ax.set_xlabel("")

    # ─────────────────────────────────────────────────────────────────────────
    # Row 2：截面图（y=0 和 x=0）
    #        左半部分：y=0 截面，右半部分：x=0 截面
    # ─────────────────────────────────────────────────────────────────────────
    mid = N_GRID // 2
    xvals = gx
    yvals = gy

    # 创建两个跨列的子图（各占一半列宽）
    ax_sy = fig.add_subplot(gs[2, :N//2] if N//2 > 0 else gs[2, :])
    ax_sx = fig.add_subplot(gs[2, N//2:])

    # 绘制精确解
    ax_sy.plot(xvals, u_exact[:, mid], "k-", lw=2.0, label="Exact")
    ax_sx.plot(yvals, u_exact[mid, :], "k-", lw=2.0, label="Exact")

    # 绘制各模型的截面
    for name in model_names:
        sty = MODEL_STYLE[name]
        ax_sy.plot(xvals, preds[name][:, mid],
                   color=sty["color"], ls=sty["ls"], lw=1.5,
                   label=sty['label'])
        ax_sx.plot(yvals, preds[name][mid, :],
                   color=sty["color"], ls=sty["ls"], lw=1.5,
                   label=sty['label'])

    ax_sy.set_title("Section $y=0$", fontsize=FS_TITLE, fontweight="bold")
    ax_sy.set_xlabel("$x$", fontsize=FS_LABEL)
    ax_sy.tick_params(labelsize=FS_TICK)
    ax_sy.legend(fontsize=FS_LEGEND, ncol=2, framealpha=0.85)
    ax_sy.grid(True, alpha=0.3)

    ax_sx.set_title("Section $x=0$", fontsize=FS_TITLE, fontweight="bold")
    ax_sx.set_xlabel("$y$", fontsize=FS_LABEL)
    ax_sx.tick_params(labelsize=FS_TICK)
    ax_sx.legend(fontsize=FS_LEGEND, ncol=2, framealpha=0.85)
    ax_sx.grid(True, alpha=0.3)

    # 保存
    for ext in ("pdf", "png"):
        path = f"{save_path.rsplit('.', 1)[0]}.{ext}" if '.' in save_path else f"{save_path}.{ext}"
        fig.savefig(path, dpi=200, bbox_inches="tight")
        print(f"Figure 2 saved: {path}")
    plt.close(fig)


# =============================================================================
# ── 主程序 ────────────────────────────────────────────────────────────────────
# =============================================================================
def main():
    print("=" * 60)
    print("  Loading convergence curves (PDE Poisson)")
    print("=" * 60)

    curves = load_curves()

    print("\n" + "=" * 60)
    print("  Loading model weights")
    print("=" * 60)
    models = {
        "PINN":      build_model_pinn(),
        "MscaleDNN": build_model_mscale(),
        "Test-A":    build_model_testA(),
        "Test-B":    build_model_testB(),
        "PFDv1":     build_model_pfdv1(),
        "PFDv2":     build_model_pfdv2(),
    }

    present = [k for k, v in models.items() if v is not None]
    missing = [k for k, v in models.items() if v is None]
    if present:
        print(f"  Available: {present}")
    if missing:
        print(f"  Missing  : {missing} (skipped)")

    print("\n" + "=" * 60)
    print("  Plotting Figure 1: Convergence curves")
    print("=" * 60)
    if curves:
        plot_figure1(curves, save_path="figure1_pde_convergence.pdf")
        plot_figure1(curves, save_path="figure1_pde_convergence.png")
    else:
        print("  No curve data found, skipping.")

    print("\n" + "=" * 60)
    print("  Plotting Figure 2: Solution visualization (Academic Style)")
    print("=" * 60)
    if any(v is not None for v in models.values()):
        plot_figure2(models, save_path="figure2_pde_solution.pdf")
        plot_figure2(models, save_path="figure2_pde_solution.png")
    else:
        print("  No model weights found, skipping.")

    print("\nDone.")


if __name__ == "__main__":
    main()